# Demo passo passo: mini DGM-Hyperagents

Questo notebook accompagna [`REPORT.md`](../REPORT.md) (relazione ufficiale; traduzione IT: [`RELAZIONE.md`](../RELAZIONE.md)) e fa girare a mano il ciclo `generate -> evaluate -> archive -> select -> modify` su un task didattico di classificazione di stringhe palindrome.

Obiettivo: vedere passo per passo cosa succede in ogni fase, capire perche' l'archivio cresce, come il meta-agent sblocca mutazioni piu' forti e come la parent selection bilancia exploitation ed exploration.

Tutto il codice riusa il modulo `experiments/common.py`.

## 1. Setup degli import

Importiamo le classi definite in `common.py`. Sono le stesse usate dalle sei demo della relazione, quindi ogni concetto qui ha un corrispondente esatto nella mini-libreria.

In [ ]:
import os, sys, random

# Permette di importare common.py quando il notebook gira da experiments/
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))

from experiments.common import (
    CandidateProgram,
    TaskAgent,
    MetaAgent,
    HyperAgent,
    ArchiveEntry,
    evaluate,
    is_valid,
    select_score_child_prop,
    run_dgm_h,
    best_by_iteration,
)

print("Import ok.")

## 2. Definizione del task

Il task e' un classificatore binario: data una stringa, ritornare `True` se e' palindroma, `False` altrimenti.

Lo dividiamo in:
- `TRAIN_DATA`: visibile al meta-agent durante l'evoluzione (usato per la selezione del parent).
- `TEST_DATA`: usato solo per misurare la generalizzazione.

In [ ]:
TRAIN_DATA = [
    ("anna", True), ("otto", True), ("radar", True),
    ("casa", False), ("python", False), ("abc", False),
    ("level", True), ("banana", False), ("abca", False),
    ("abda", False), ("axya", False), ("racecar", True),
]

TEST_DATA = [
    ("kayak", True), ("madam", True), ("chatgpt", False),
    ("robot", False), ("noon", True), ("openai", False),
    ("abcda", False), ("abxba", True), ("acxca", True), ("axcya", False),
]
print(f"Training: {len(TRAIN_DATA)} esempi, Test: {len(TEST_DATA)} esempi")

## 3. Costruzione dell'agente iniziale

Un `CandidateProgram` e' un programma Python sotto forma di stringa. La sua funzione `compile()` esegue il codice e restituisce la funzione `solve(x)`.

Partiamo da un agente ingenuo che risponde sempre `False`.

In [ ]:
initial_program = CandidateProgram(
    name="initial_false",
    description="Agente iniziale ingenuo: risponde sempre False.",
    code="def solve(x):\n    return False\n",
)

initial_task_agent = TaskAgent(program=initial_program)

print("Codice:")
print(initial_program.code)
print("Score train iniziale:", evaluate(HyperAgent(initial_task_agent, MetaAgent([]), 0), TRAIN_DATA))

## 4. Costruzione del meta-agent

Il meta-agent ha due tipi di mutazioni:

- `base_candidates`: sempre disponibili (strategie di test, euristiche parziali).
- `hidden_candidates`: sbloccate solo se l'archivio raggiunge `unlock_score_threshold`.

Quando una mutazione viene sbloccata, l'evento viene registrato in `mutation_memory`. Questo simula la "persistent memory" descritta in Sec. 5.2 dell'articolo HyperAgents.

In [ ]:
base_candidates = [
    CandidateProgram("always_true", "Costante True.", "def solve(x):\n    return True\n"),
    CandidateProgram("always_false", "Costante False.", "def solve(x):\n    return False\n"),
    CandidateProgram(
        "first_last_equal",
        "Euristica parziale: primo e ultimo carattere uguali.",
        "def solve(x):\n    return True if not x else x[0] == x[-1]\n",
    ),
]

hidden_candidates = [
    CandidateProgram(
        "reverse_check",
        "Strategia corretta: confronto con la stringa invertita.",
        "def solve(x):\n    s = x.strip().lower()\n    return s == s[::-1]\n",
    )
]

meta_agent = MetaAgent(
    base_candidates=base_candidates,
    hidden_candidates=hidden_candidates,
    self_improve=True,
    unlock_score_threshold=0.75,
)
print("Base candidates:", [c.name for c in meta_agent.base_candidates])
print("Hidden candidates:", [c.name for c in meta_agent.hidden_candidates])

## 5. Costruzione dell'hyperagent e inizializzazione dell'archivio

L'`HyperAgent` combina task agent e meta-agent in un unico oggetto modificabile. L'archivio e' una lista di `ArchiveEntry`, una per ogni nodo della popolazione.

In [ ]:
initial_hyperagent = HyperAgent(
    task_agent=initial_task_agent,
    meta_agent=meta_agent,
    generation=0,
)

archive = [
    ArchiveEntry(
        node_id=0,
        parent_id=None,
        iteration=0,
        agent=initial_hyperagent,
        train_score=evaluate(initial_hyperagent, TRAIN_DATA),
        test_score=evaluate(initial_hyperagent, TEST_DATA),
        valid=is_valid(initial_hyperagent),
        seed=0,
        variant="notebook_demo",
    )
]

print(
    f"Archivio iniziale: {len(archive)} elemento\n"
    f"Nome: {archive[0].name}\n"
    f"Train: {archive[0].train_score:.2f}  Test: {archive[0].test_score:.2f}"
)

## 6. Un'iterazione del ciclo, spiegata

Eseguiamo manualmente una singola iterazione, stampando ogni passaggio. Cosi' si vede:

1. quale parent viene scelto dall'archivio;
2. quali candidati il meta-agent propone;
3. il figlio generato e la sua valutazione;
4. l'aggiornamento dell'archivio.

In [ ]:
rng = random.Random(0)

parent_entry = select_score_child_prop(archive, rng)
print(f"[1] Parent scelto: id={parent_entry.node_id} ({parent_entry.name}), score={parent_entry.train_score:.2f}")

candidates = parent_entry.agent.meta_agent.generate_candidates(parent_entry.agent, archive)
print("[2] Candidati proposti dal meta-agent:")
for c in candidates:
    print(f"    - {c.name}: {c.description}")

children = parent_entry.agent.modify(archive, rng, candidates_per_iteration=1)
child = children[0]
train = evaluate(child, TRAIN_DATA)
test = evaluate(child, TEST_DATA)
print(f"[3] Figlio generato: {child.task_agent.program.name} | train={train:.2f}, test={test:.2f}")

archive.append(
    ArchiveEntry(
        node_id=len(archive),
        parent_id=parent_entry.node_id,
        iteration=1,
        agent=child,
        train_score=train,
        test_score=test,
        valid=is_valid(child),
        seed=0,
        variant="notebook_demo",
    )
)
print(f"[4] Archivio ora ha {len(archive)} elementi")

## 7. Ciclo completo con `run_dgm_h`

Ora eseguiamo l'intero ciclo (10 iterazioni) usando la funzione di alto livello. La storia restituita contiene anche l'agente iniziale, quindi possiamo ricostruire l'andamento del best test score nel tempo.

In [ ]:
fresh_meta = MetaAgent(
    base_candidates=base_candidates,
    hidden_candidates=hidden_candidates,
    self_improve=True,
    unlock_score_threshold=0.75,
)

history = run_dgm_h(
    variant="notebook_demo",
    train_data=TRAIN_DATA,
    test_data=TEST_DATA,
    initial_program=initial_program,
    meta_agent=fresh_meta,
    selector=select_score_child_prop,
    iterations=10,
    seed=42,
    archive_mode="full",
)
print(f"Nodi nell'archivio: {len(history)}")
best = max(history, key=lambda e: e.train_score)
print(f"Miglior agente: {best.name} (train={best.train_score:.2f}, test={best.test_score:.2f})")
print("Memoria meta-agent del miglior agente:")
for note in best.agent.meta_agent.mutation_memory:
    print(" -", note)

## 8. Visualizzazione dell'archivio

Stampiamo tutti i nodi dell'archivio in tabella, evidenziando il rapporto genealogico. Da qui si vede chiaramente perche' l'archivio e' un albero di stepping stones.

In [ ]:
print(f"{'id':>3} {'parent':>6} {'iter':>4} {'train':>6} {'test':>6}  agente")
print("-" * 60)
for entry in history:
    parent = entry.parent_id if entry.parent_id is not None else "-"
    print(
        f"{entry.node_id:>3} {str(parent):>6} {entry.iteration:>4} "
        f"{entry.train_score:>6.2f} {entry.test_score:>6.2f}  {entry.name}"
    )

## 9. Curva di progresso del best test score

Costruiamo il "best so far" per iterazione e lo mostriamo. E' la stessa metrica usata nei plot delle demo della relazione.

In [ ]:
import matplotlib.pyplot as plt

rows = best_by_iteration(history)
xs = [r["iteration"] for r in rows]
ys = [r["best_test"] for r in rows]

plt.figure(figsize=(7, 4))
plt.plot(xs, ys, marker="o", linewidth=2)
plt.title("Best test score per iterazione (demo passo passo)")
plt.xlabel("Iterazione")
plt.ylabel("Best test score")
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Conclusione

Questo notebook ha mostrato:

- come e' definito un task agent (codice Python in una stringa);
- come il meta-agent espone mutazioni di base e mutazioni nascoste sbloccabili;
- come l'hyperagent unisce i due ruoli in un singolo oggetto modificabile;
- come una singola iterazione del ciclo aggiorna l'archivio;
- come la funzione `run_dgm_h` esegue tutto Algorithm 1 dell'articolo.

Per gli esperimenti completi (ablation, parent selection, secondo dominio, code-repair) vedere i singoli script in `experiments/` e [`REPORT.md`](../REPORT.md) (traduzione IT: [`RELAZIONE.md`](../RELAZIONE.md)).